# What ``@entities`` and a resource's ``@meta`` refuse

Every class ``@entities`` lists must be an ``@entity``, and it is listed once. Only a resource declares ``@entities``. A resource's ``@meta`` takes no domain. Each refusal happens when the class is declared and names what is wrong.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

In [ ]:
!pip install -q aoa-action-machine

In [2]:
from pydantic import BaseModel, Field

from aoa.action_machine.domain import BaseCoreEntity
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.exceptions import DecoratorArgumentTypeError, EntitiesDecoratorError
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources import BaseStorage

## 2. Declare the model

In [3]:
class StoreDomain(BaseDomain):
    """Group the shop's orders."""

    name = "store"
    description = "Store"


@entity(description="Customer order", domain=StoreDomain)
class OrderCoreEntity(BaseCoreEntity):
    """An order of the store."""

    id: str = Field(description="Order identifier")


class OrderRow(BaseModel):
    """A plain model: not an entity of the domain."""

    id: str


class ReportBuilder:
    """A plain class: not a resource."""

## 3. Run and check the observation

In [4]:
try:

    @entities(OrderRow)
    class RowsStorage(BaseStorage):
        """Lists a plain model."""

except EntitiesDecoratorError as error:
    print("not an entity:", error)

try:

    @entities(OrderCoreEntity, OrderCoreEntity)
    class TwiceStorage(BaseStorage):
        """Lists one entity twice."""

except EntitiesDecoratorError as error:
    print("listed twice:", error)

try:
    entities(OrderCoreEntity)(ReportBuilder)
except EntitiesDecoratorError as error:
    print("not a resource:", error)

try:

    @meta(description="The shop database", domain=StoreDomain)
    class ShopDatabaseStorage(BaseStorage):
        """Names a domain, as a resource no longer does."""

except DecoratorArgumentTypeError as error:
    print("a resource with a domain:", error)

not an entity: @entities: 'OrderRow' is not an @entity. Every @entities argument must be a class decorated with @entity(description=..., domain=...).
listed twice: @entities: class 'OrderCoreEntity' is listed twice. A database lists each of its entities once.
not a resource: @entities applies only to a resource (a BaseResource subclass); 'ReportBuilder' is not one. A resource is a database, and it lists the entities that lie in it.
a resource with a domain: @meta on ShopDatabaseStorage: a resource is a database shared by domains and takes no domain — its domains are the domains of the entities it lists with @entities. Remove domain=.


## 4. Explain the result

Each declaration is refused when the class is declared, naming what is wrong: a listed class that is not an `@entity`, a class listed twice, `@entities` on a class that is not a resource, and a domain on a resource's `@meta`.